# Proyecto 5: Segmentación de Clientes mediante Clustering
## Notebook 2: Limpieza de Datos y Estandarización

### Objetivo
Preparar los datos para el clustering mediante:
1. Detección y tratamiento de outliers
2. Análisis de distribuciones
3. Estandarización de variables
4. Generación de conjunto de datos listo para clustering

### Importancia
El clustering es sensible a la escala de las variables. Una variable con valores grandes puede dominar la distancia euclidiana. Por ello, estandarizamos.

## 1. Importación de Librerías y Carga de Datos

In [ ]:
# Librerías de manipulación de datos
import pandas as pd
import numpy as np

# Librerías de visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Librerías de preprocesamiento
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.decomposition import PCA

# Configuración
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
warnings_filter = 'ignore'

import warnings
warnings.filterwarnings(warnings_filter)

print("✓ Librerías importadas correctamente")

In [ ]:
# Cargar el dataset
ruta_datos = '../data/Wholesale customers data.csv'
df = pd.read_csv(ruta_datos)

print(f"✓ Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"\nPrimeras filas:")
df.head()

## 2. Análisis de Outliers

### Métodos para detectar outliers:
- **IQR (Rango Intercuartílico)**: outlier si valor < Q1-1.5*IQR o > Q3+1.5*IQR
- **Z-score**: outlier si |z| > 3
- **Método de Tukey**: Similar a IQR

In [ ]:
# Función para detectar outliers usando IQR
def detectar_outliers_iqr(df, columnas):
    outliers = {}
    for col in columnas:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        limite_inferior = Q1 - 1.5 * IQR
        limite_superior = Q3 + 1.5 * IQR
        
        outliers_col = df[(df[col] < limite_inferior) | (df[col] > limite_superior)].index.tolist()
        outliers[col] = len(outliers_col)
    
    return outliers

# Variables numéricas (características de gasto)
variables_numericas = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']

# Detectar outliers
outliers_detectados = detectar_outliers_iqr(df, variables_numericas)

print("Número de outliers detectados por variable (método IQR):")
for col, count in outliers_detectados.items():
    pct = (count / len(df)) * 100
    print(f"  • {col}: {count} ({pct:.2f}%)")

## 3. Visualización de Distribuciones y Outliers

In [ ]:
# Boxplot para visualizar outliers
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.ravel()

for idx, col in enumerate(variables_numericas):
    axes[idx].boxplot(df[col], vert=True)
    axes[idx].set_title(f'Boxplot: {col}', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Valor (Unidades Monetarias)')
    axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Boxplots generados para visualización de outliers")

## 4. Histogramas de Distribuciones Originales

In [ ]:
# Histogramas de las variables originales
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.ravel()

for idx, col in enumerate(variables_numericas):
    axes[idx].hist(df[col], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribución: {col}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Valor')
    axes[idx].set_ylabel('Frecuencia')
    axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Histogramas de distribuciones generados")

## 5. Decisión sobre Tratamiento de Outliers

**Consideraciones:**
- En clustering, los outliers pueden formar clusters propios y son informativos
- Los métodos de clustering avanzados (DBSCAN) pueden identificarlos como ruido
- La estandarización es más importante que la eliminación
- **Decisión:** Mantendremos los outliers pero usaremos estandarización robusta si es necesario

In [ ]:
# Crear una copia para trabajar
df_procesado = df.copy()

print("Estados de decisión sobre outliers:")
print("✓ Los outliers se MANTIENEN para análisis de clustering")
print("  Razón: Pueden representar segmentos de mercado importantes")
print(f"\nDataset de trabajo: {df_procesado.shape[0]} filas, {df_procesado.shape[1]} columnas")

## 6. Estandarización de Variables

### Métodos de Estandarización:

1. **StandardScaler (Z-score normalization)**
   - $z = \frac{x - \mu}{\sigma}$
   - Media = 0, Desv. Estándar = 1
   - Sensible a outliers

2. **MinMaxScaler (Normalización Min-Max)**
   - $x_{scaled} = \frac{x - min}{max - min}$
   - Rango [0, 1]
   - Preserva la forma de la distribución

3. **RobustScaler**
   - Usa mediana y rango intercuartílico
   - Robusto ante outliers

In [ ]:
# Crear escaladores
scaler_standard = StandardScaler()
scaler_minmax = MinMaxScaler()
scaler_robust = RobustScaler()

# Aplicar estandarización Z-score (StandardScaler)
df_estandarizado = df_procesado.copy()
df_estandarizado[variables_numericas] = scaler_standard.fit_transform(df_procesado[variables_numericas])

print("✓ Estandarización Z-score aplicada (StandardScaler)")
print(f"\nEstadísticas después de estandarización:")
print(df_estandarizado[variables_numericas].describe().round(4))

In [ ]:
# Crear versión normalizada (0-1)
df_normalizado = df_procesado.copy()
df_normalizado[variables_numericas] = scaler_minmax.fit_transform(df_procesado[variables_numericas])

print("✓ Normalización Min-Max aplicada (escala 0-1)")
print(f"\nEstadísticas después de normalización:")
print(df_normalizado[variables_numericas].describe().round(4))

## 7. Comparación Visual: Antes vs Después

In [ ]:
# Comparación visual
fig, axes = plt.subplots(3, 2, figsize=(14, 10))

# Seleccionar dos variables para ejemplo
var_ejemplo1 = 'Fresh'
var_ejemplo2 = 'Milk'

# Original
axes[0, 0].hist(df_procesado[var_ejemplo1], bins=30, color='#FF6B6B', alpha=0.7, edgecolor='black')
axes[0, 0].set_title(f'Original: {var_ejemplo1}', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Frecuencia')

axes[0, 1].hist(df_procesado[var_ejemplo2], bins=30, color='#4ECDC4', alpha=0.7, edgecolor='black')
axes[0, 1].set_title(f'Original: {var_ejemplo2}', fontsize=11, fontweight='bold')

# Estandarizado
axes[1, 0].hist(df_estandarizado[var_ejemplo1], bins=30, color='#FF6B6B', alpha=0.7, edgecolor='black')
axes[1, 0].set_title(f'Estandarizado: {var_ejemplo1}', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Frecuencia')

axes[1, 1].hist(df_estandarizado[var_ejemplo2], bins=30, color='#4ECDC4', alpha=0.7, edgecolor='black')
axes[1, 1].set_title(f'Estandarizado: {var_ejemplo2}', fontsize=11, fontweight='bold')

# Normalizado
axes[2, 0].hist(df_normalizado[var_ejemplo1], bins=30, color='#FF6B6B', alpha=0.7, edgecolor='black')
axes[2, 0].set_title(f'Normalizado: {var_ejemplo1}', fontsize=11, fontweight='bold')
axes[2, 0].set_ylabel('Frecuencia')

axes[2, 1].hist(df_normalizado[var_ejemplo2], bins=30, color='#4ECDC4', alpha=0.7, edgecolor='black')
axes[2, 1].set_title(f'Normalizado: {var_ejemplo2}', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Comparación visual generada")

## 8. Matriz de Correlación - Datos Estandarizados

In [ ]:
# Calcular matriz de correlación
correlacion = df_estandarizado[variables_numericas].corr()

# Visualizar matriz de correlación
plt.figure(figsize=(10, 8))
sns.heatmap(correlacion, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, fmt='.2f')
plt.title('Matriz de Correlación - Variables de Gasto', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("✓ Matriz de correlación calculada")

## 9. Dataset Final Listo para Clustering

In [ ]:
# Seleccionar el dataset estandarizado para clustering
# Usaremos StandardScaler (Z-score) como método estándar
df_clustering = df_estandarizado.copy()

print("="*70)
print("DATASET PREPARADO PARA CLUSTERING")
print("="*70)
print(f"\n✓ Dimensiones: {df_clustering.shape[0]} muestras × {df_clustering.shape[1]} variables")
print(f"\n✓ Método de estandarización: StandardScaler (Z-score)")
print(f"\nPrimeras 5 filas del dataset estandarizado:")
print(df_clustering.head())

print(f"\n✓ Estadísticas del dataset estandarizado:")
print(df_clustering.describe().round(4))

## 10. Información de Escaladores para Referencia

In [ ]:
# Mostrar parámetros del escalador para referencia
print("\nPARÁMETROS DEL ESCALADOR ESTÁNDAR (StandardScaler):")
print("\nMedia (μ) - utilizada en estandarización:")
media_dict = dict(zip(variables_numericas, scaler_standard.mean_.round(2)))
for var, val in media_dict.items():
    print(f"  • {var}: {val}")

print("\nDesviación Estándar (σ) - utilizada en estandarización:")
std_dict = dict(zip(variables_numericas, scaler_standard.scale_.round(2)))
for var, val in std_dict.items():
    print(f"  • {var}: {val}")

print("\n" + "="*70)

## 11. Resumen del Procesamiento

In [ ]:
resumen = f"""
RESUMEN DEL PROCESAMIENTO DE DATOS
{'='*70}

1. DATOS ORIGINALES:
   • Filas: {df.shape[0]}
   • Columnas: {df.shape[1]}
   • Valores faltantes: {df.isnull().sum().sum()}
   • Outliers detectados (IQR): {sum(outliers_detectados.values())}

2. PROCESAMIENTO REALIZADO:
   ✓ Análisis de outliers con método IQR
   ✓ Outliers mantenidos (son informativos para clustering)
   ✓ Estandarización con StandardScaler (Z-score)
   ✓ Verificación de correlaciones entre variables

3. DATOS FINALES PARA CLUSTERING:
   • Filas: {df_clustering.shape[0]}
   • Columnas: {df_clustering.shape[1]}
   • Media de variables: ≈ 0
   • Desv. Estándar: ≈ 1
   • Escala: Comparable entre variables

4. PRÓXIMO PASO:
   → Análisis Exploratorio de Datos (EDA)
   → Evaluación de número óptimo de clusters
   → Aplicación de algoritmos de clustering

{'='*70}
"""

print(resumen)

In [ ]:
# Guardar información importante para próximos notebooks
print("✓ Limpieza y estandarización completada")
print("✓ El dataset está listo para Análisis Exploratorio de Datos")